In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# import pandas as pd
# data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# data.to_csv("submission.csv", index=False)

**Import modules**

In [ ]:
import os
import inspect
import numpy as np
import pandas as pd
import torch

from dataclasses import dataclass
from typing import Optional, Union

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

In [ ]:
INPUT_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-large"  
MAX_LEN = 256
BATCH_SIZE = 2          
GRAD_ACCUM = 8          
EPOCHS = 3
LR = 1e-5
SEED = 42

OPTIONS = ["A", "B", "C", "D", "E"]
option_to_index = {o: i for i, o in enumerate(OPTIONS)}
index_to_option = {i: o for i, o in enumerate(OPTIONS)}

In [ ]:
train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
print(train_df.head())

# fill any empty text so tokenizer never gets NaN
for col in ["prompt"] + OPTIONS:
    train_df[col] = train_df[col].fillna(" ").astype(str)
    test_df[col] = test_df[col].fillna(" ").astype(str)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess_train(example):
    first = [example["prompt"]] * 5
    second = [example[opt] for opt in OPTIONS]
    tokenized = tokenizer(first, second, truncation=True, max_length=MAX_LEN)
    tokenized["label"] = option_to_index[example["answer"].strip()]
    return tokenized


def preprocess_test(example):
    first = [example["prompt"]] * 5
    second = [example[opt] for opt in OPTIONS]
    tokenized = tokenizer(first, second, truncation=True, max_length=MAX_LEN)
    return tokenized

In [ ]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        has_labels = label_name in features[0]
        labels = [f.pop(label_name) for f in features] if has_labels else None

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened = [
            [{k: v[i] for k, v in f.items()} for i in range(num_choices)]
            for f in features
        ]
        flattened = sum(flattened, [])

        batch = self.tokenizer.pad(
            flattened,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch



In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    top3 = np.argsort(-logits, axis=1)[:, :3]
    score = 0.0
    for i, label in enumerate(labels):
        for rank in range(3):
            if top3[i][rank] == label:
                score += 1.0 / (rank + 1)   # 1, 1/2 or 1/3
                break
    return {"map@3": score / len(labels)}

In [ ]:
train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_val = max(int(len(train_df) * 0.1), 1)
val_split = train_df.iloc[:n_val].reset_index(drop=True)
train_split = train_df.iloc[n_val:].reset_index(drop=True)

train_ds = Dataset.from_pandas(train_split)
val_ds = Dataset.from_pandas(val_split)
test_ds = Dataset.from_pandas(test_df)

train_tok = train_ds.map(preprocess_train, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess_train, remove_columns=val_ds.column_names)
test_tok = test_ds.map(preprocess_test, remove_columns=test_ds.column_names)


In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

# newer transformers renamed evaluation_strategy -> eval_strategy; support both
eval_kw = "eval_strategy" if "eval_strategy" in inspect.signature(
    TrainingArguments.__init__).parameters else "evaluation_strategy"

args = TrainingArguments(**{
    "output_dir": "output",
    eval_kw: "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "map@3",
    "greater_is_better": True,
    "learning_rate": LR,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "num_train_epochs": EPOCHS,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "fp16": True,
    "report_to": "none",
    "seed": SEED,
    "save_total_limit": 1,
})

# newer transformers renamed tokenizer -> processing_class in Trainer; support both
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

trainer.train()
print("Validation:", trainer.evaluate())

In [ ]:
test_logits = trainer.predict(test_tok).predictions       # (n_test, 5)
top3_idx = np.argsort(-test_logits, axis=1)[:, :3]         # best 3 per question

predictions = [
    " ".join(index_to_option[i] for i in row)              # e.g. "A B C"
    for row in top3_idx
]

submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": predictions,
})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("submission.csv created successfully")

**Milestone-1**

In [ ]:
# importing libraries

import string
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import os
import pandas as pd

DATA_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge"

print("Files present:")
print(os.listdir(DATA_PATH))

train_path = os.path.join(DATA_PATH, "train.csv")
test_path = os.path.join(DATA_PATH, "test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

df = train_df

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print(train_df.columns.tolist())
print(test_df.columns.tolist())


OPTIONS = ["A", "B", "C", "D", "E"]




# Q1: Frequency of correct answers (sum of max + min counts)

answer_counts = df["answer"].value_counts()
print("\nQ1 - Answer frequency distribution:")
print(answer_counts)

most_freq = answer_counts.max()
least_freq = answer_counts.min()
q1_answer = most_freq + least_freq
print(f"Most frequent count: {most_freq}")
print(f"Least frequent count: {least_freq}")
print(f"ANSWER Q1: {q1_answer}")

# Q2: Unique words in cleaned prompt column

def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()

all_words = set()
for prompt in df["prompt"]:
    all_words.update(clean_text(prompt))

q2_answer = len(all_words)
print(f"\nQ2 - Unique words in cleaned prompts: {q2_answer}")
print(f"ANSWER Q2: {q2_answer}")

# Q3: Row ID 1 - words left after removing stop 

row1_words = clean_text(df.loc[df["id"] == 1, "prompt"].values[0])
row1_filtered = [w for w in row1_words if w not in ENGLISH_STOP_WORDS]
q3_answer = len(row1_filtered)
print(f"\nQ3 - Row 1 words after stop word removal: {q3_answer}")
print(f"Cleaned words: {row1_words}")
print(f"After filtering: {row1_filtered}")
print(f"ANSWER Q3: {q3_answer}")

# Q4: TF-IDF vocabulary size (prompt + all options combined per row)

combined_docs = []
for _, row in df.iterrows():
    text = str(row["prompt"])
    for opt in OPTIONS:
        text += " " + str(row[opt])
    combined_docs.append(text)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_docs)
q4_answer = len(vectorizer.get_feature_names_out())
print(f"\nQ4 - TF-IDF vocabulary size: {q4_answer}")
print(f"ANSWER Q4: {q4_answer}")


# Q5: Cosine similarity prompt vs option A for Row ID 1

row1 = df[df["id"] == 1].iloc[0]
prompt_vec = vectorizer.transform([str(row1["prompt"])])
option_a_vec = vectorizer.transform([str(row1["A"])])
sim_score = cosine_similarity(prompt_vec, option_a_vec)[0][0]
q5_answer = round(sim_score, 4)
print(f"\nQ5 - Cosine similarity (prompt vs A) Row 1: {q5_answer}")
print(f"ANSWER Q5: {q5_answer}")

# Q6: percentage of rows where highest cosine similarity option = correct answer

correct_matches = 0
total_rows = len(df)

for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row["prompt"])])
    best_opt = None
    best_sim = -1

    for opt in OPTIONS:
        opt_vec = vectorizer.transform([str(row[opt])])
        sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        if sim > best_sim:
            best_sim = sim
            best_opt = opt

    if best_opt == row["answer"]:
        correct_matches += 1

q6_answer = round((correct_matches / total_rows) * 100, 2)
print(f"\nQ6 - Highest similarity matches correct answer: {correct_matches}/{total_rows}")
print(f"ANSWER Q6: {q6_answer}%")


# Q7: MAP@3 for ground truth C, prediction C A B

def map_at_k(true_answer, predictions, k=3):
    """MAP@k for single-label MCQ (one correct answer)."""
    preds = predictions[:k]
    if true_answer in preds:
        rank = preds.index(true_answer) + 1
        return 1.0 / rank
    return 0.0

q7_answer = map_at_k("C", ["C", "A", "B"])
print(f"\nQ7 - MAP@3 (truth=C, pred=C A B): {q7_answer}")
print(f"ANSWER Q7: {q7_answer}")


# Q8: MAP@3 for ground truth B, prediction D B E

q8_answer = map_at_k("B", ["D", "B", "E"])
print(f"\nQ8 - MAP@3 (truth=B, pred=D B E): {q8_answer}")
print(f"ANSWER Q8: {q8_answer}")


# Q9: Majority Class Baseline MAP@3

top3_answers = answer_counts.index[:3].tolist()
print(f"\nQ9 - Top 3 most frequent answers: {top3_answers}")

map_scores = []
for _, row in df.iterrows():
    score = map_at_k(row["answer"], top3_answers)
    map_scores.append(score)

q9_answer = round(sum(map_scores) / len(map_scores), 4)
print(f"ANSWER Q9: {q9_answer}")


# Q10: TF-IDF Pipeline MAP@3 on full training set

pipeline_scores = []

for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row["prompt"])])
    sims = []

    for opt in OPTIONS:
        opt_vec = vectorizer.transform([str(row[opt])])
        sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        sims.append((opt, sim))

    # Sort by similarity (highest first), take top 3
    sims.sort(key=lambda x: x[1], reverse=True)
    top3_pred = [opt for opt, _ in sims[:3]]

    score = map_at_k(row["answer"], top3_pred)
    pipeline_scores.append(score)

q10_answer = round(sum(pipeline_scores) / len(pipeline_scores), 4)
print(f"\nQ10 - TF-IDF Pipeline MAP@3: {q10_answer}")
print(f"ANSWER Q10: {q10_answer}")

In [ ]:
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


In [ ]:
# print(train_df.shape)
# print(test_df.shape)

In [ ]:
# print(train_df.columns.tolist())
# print(test_df.columns.tolist())